# Olist E-Commerce — Data Cleaning

Two categories of cleaning happen here:
1. Filling nulls in `products_df` where a null represents genuinely missing product metadata (not a missing transaction).
2. Converting date/timestamp columns from string to `datetime`.

No nulls are filled in `orders_df` or `reviews_df` — a missing delivery date or missing review comment is a real, valid state (the order wasn't delivered yet / the reviewer left no comment), not something that should be fabricated.

> **Paths:** this notebook uses relative paths (`../data/raw/...`, `../data/cleaned/...`) and assumes it is opened and run from inside the `01_data_cleaning/` folder, as laid out in this repo. If you move the notebook elsewhere, update the paths accordingly.

In [1]:
import pandas as pd

## Load raw datasets

In [2]:
customers_df = pd.read_csv(r"../data/raw/olist_customers_dataset.csv")
geolocation_df = pd.read_csv(r"../data/raw/olist_geolocation_dataset.csv")
order_items_df = pd.read_csv(r"../data/raw/olist_order_items_dataset.csv")
payments_df = pd.read_csv(r"../data/raw/olist_order_payments_dataset.csv")
reviews_df = pd.read_csv(r"../data/raw/olist_order_reviews_dataset.csv")
orders_df = pd.read_csv(r"../data/raw/olist_orders_dataset.csv")
products_df = pd.read_csv(r"../data/raw/olist_products_dataset.csv")
sellers_df = pd.read_csv(r"../data/raw/olist_sellers_dataset.csv")
category_translation_df = pd.read_csv(r"../data/raw/product_category_name_translation.csv")

## Fill nulls in `products_df`
Approximately 610 products lack category and dimension metadata. Rather than forcing arbitrary imputations like 'N/A' or 0 (which would artificially skew downstream dimensional averages in SQL), these are intentionally preserved as native NaNs. They will be imported as true NULLs into the SQL Server database to be handled safely by aggregate functions.

## Convert date columns to datetime

All date/timestamp columns arrive from the CSVs as plain strings. Converting them here with `pd.to_datetime` in place — **not** into a new, separately-named column — since assigning the converted result to a new column name previously caused a duplicate typo'd column to end up in the database (`order_purchse_timestamp`), which then had to be cleaned up separately in SQL. Converting in place avoids that entirely.

Note: because these dataframes get written out to CSV below, and CSV has no way to store column dtype metadata, this datetime typing does not survive the round trip — SQL Server sees these as text again on import and the columns are recast to `DATETIME` there instead (see `02_checking_datetime_data_type.sql` / `02.1_changing_varchar_to_datetime.sql`). The conversion here is still worth doing, since it's needed for the date-logic checks done during EDA and keeps this notebook self-consistent, but it is not what ultimately gives the SQL tables their datetime types.

In [3]:
orders_df['order_purchase_timestamp'] = pd.to_datetime(orders_df['order_purchase_timestamp'])
orders_df['order_approved_at'] = pd.to_datetime(orders_df['order_approved_at'])
orders_df['order_delivered_carrier_date'] = pd.to_datetime(orders_df['order_delivered_carrier_date'])
orders_df['order_delivered_customer_date'] = pd.to_datetime(orders_df['order_delivered_customer_date'])
orders_df['order_estimated_delivery_date'] = pd.to_datetime(orders_df['order_estimated_delivery_date'])

In [4]:
reviews_df['review_creation_date'] = pd.to_datetime(reviews_df['review_creation_date'])
reviews_df['review_answer_timestamp'] = pd.to_datetime(reviews_df['review_answer_timestamp'])

In [5]:
order_items_df['shipping_limit_date'] = pd.to_datetime(order_items_df['shipping_limit_date'])

## Export cleaned datasets

In [6]:
customers_df.to_csv(r"../data/cleaned/customers_clean.csv", index=False)
geolocation_df.to_csv(r"../data/cleaned/geolocation_clean.csv", index=False)
order_items_df.to_csv(r"../data/cleaned/order_items_clean.csv", index=False)
payments_df.to_csv(r"../data/cleaned/payments_clean.csv", index=False)
reviews_df.to_csv(r"../data/cleaned/reviews_clean.csv", index=False)
orders_df.to_csv(r"../data/cleaned/orders_clean.csv", index=False)
products_df.to_csv(r"../data/cleaned/products_clean.csv", index=False)
sellers_df.to_csv(r"../data/cleaned/sellers_clean.csv", index=False)
category_translation_df.to_csv(r"../data/cleaned/category_translation.csv", index=False)